In [1]:
import os, sys
import copy as cp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
import math
import collections
import pandas as pd
import pickle

from datetime import date

In [2]:
sys.path.append(os.path.abspath('../code/'))

from RMF import RMF

In [3]:
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['axes.titlesize'] = 8    # Size of the plot title
plt.rcParams['axes.labelsize'] = 8    # Size of x and y labels
plt.rcParams['xtick.labelsize'] = 7   # Size of x-tick labels
plt.rcParams['ytick.labelsize'] = 7   # Size of y-tick labels
plt.rcParams['legend.fontsize'] = 7
plt.rcParams['font.size'] = 8

plt.rcParams.update({
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'xtick.minor.size': 2,
    'ytick.minor.size': 2,
    'xtick.major.pad': 2,
    'ytick.major.pad': 2,
    'axes.linewidth': 0.8,
})

plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'serif'  # Or 'serif' depending on your main font
plt.rcParams['mathtext.it'] = 'serif:italic'
plt.rcParams['mathtext.bf'] = 'serif:bold'

cm_to_inch = 2.54

In [4]:
save = True

outputdir = os.path.join ('../results/RMF', str (date.today ()))
os.makedirs(outputdir, exist_ok=True)

In [5]:
def rmf_simulation_wrapper (nsims, M, sigma_additive, sigma_hoc, epsilon,
                            weighted=False, top=None,
                            nbatch=5000, seed=None, scenarios=['adaptive','conditional','random','beneficial']) :

    ntypes = len (scenarios)
    R2     = np.zeros ((ntypes, nsims, M+1))
    nMax   = np.zeros ((ntypes, nsims, M+1))
    nPaths = np.zeros ((ntypes, nsims))
    Vy     = np.zeros ((ntypes, nsims, M+1))
    
    if 'adaptive' in scenarios and 'conditional' in scenarios :
        Muts = np.zeros ((2, nsims, M))
    else :
        Muts = np.zeros ((1, nsims, M))

    Add = np.zeros ((ntypes, nsims, 2**M))
    Eps = np.zeros ((ntypes, nsims, 2**M))
    Ys  = np.zeros_like (Add)

    # create rng
    if seed is not None :
        my_rng = np.random.default_rng (seed)
    else :
        my_rng = np.random.default_rng ()

    i = 0
    while i < nsims :
        if i % 20 == 0 :
            print ('Starting simulation: ' + str (i))
       
        try :
            Landscape = RMF (L=L, sigma_add=sigma_additive, sigma_hoc=sigma_hoc, rng=my_rng)
            Landscape.generate_libraries (M, scenarios=scenarios, epsilon=epsilon, weighted=weighted)
            Landscape.compute_statistics ()
            R2[:,i,:], nMax[:,i,:], nPaths[:,i], Vy[:,i,:] = Landscape.get_statistics (scenarios=scenarios)
            for si in range (len (scenarios)) :
                scen = scenarios[si]
                Add[si,i,:] = cp.deepcopy (Landscape.ascertained[scen]['A'])
                Eps[si,i,:] = cp.deepcopy (Landscape.ascertained[scen]['E'])
                Ys[si,i,:]  = cp.deepcopy (Landscape.ascertained[scen]['y'])
            
            if nPaths[0,i] == 0 :
                print (Landscape.ascertained['adaptive'])
                Xw = Landscape.ascertained['adaptive']['X']
                yw = Landscape.ascertained['adaptive']['y']

            if 'adaptive' in scenarios :
                Muts[0,i,:] = cp.deepcopy (Landscape.ascertained['adaptive']['mutations'])
            if 'conditional' in scenarios :
                Muts[1,i,:] = cp.deepcopy (Landscape.ascertained['conditional']['mutations'])

            if i == 0 :
                Xout = cp.deepcopy (Landscape.ascertained[scenarios[0]]['X'])
                
            i += 1
        
        except Exception as e :
            print (e)
            print ('failure: ' + str (i))
            i += 0

    return R2, nMax, nPaths, Vy, Add, Eps, Ys, Xout, Muts

In [6]:
# set seed
seed = 98763111

# parameters
L = 100
M = 4
c_error  = 10**(-3)  # conditional error threshold
weighted = False     # uniform sampling among beneficial mutations
sigma_a  = 1         # additive variance
nsims    = 100

# different scenarios
scenarios = ['adaptive','random','beneficial', 'deleterious','conditional']
colors    = ['salmon', 'lightgray', 'gold', 'cornflowerblue','dimgray']
ntypes    = len (scenarios)

# sigma epsilon values
sigma_es = np.concatenate ([np.logspace (-1,0,5), np.logspace (0,1,6)[1:]])
np.savetxt (os.path.join (outputdir, 'sigmas.txt'), sigma_es)

# to save files
outstring = str (M) + '_' + str (L) + '_' + str (nsims)

In [7]:
for ep in sigma_es :
    epsilon_e = c_error * np.sqrt ((L * sigma_a**2) + ep**2)
    print (epsilon_e)

0.010000499987500626
0.010001581013849844
0.01000499875062461
0.010015798908026102
0.01004987562112089
0.010124815377650576
0.010310653395629294
0.010763314170115595
0.011824158196478501
0.01414213562373095


In [8]:
print (sigma_es)

[ 0.1         0.17782794  0.31622777  0.56234133  1.          1.58489319
  2.51188643  3.98107171  6.30957344 10.        ]


In [9]:
# dictionary to store results for different hoc components
SigmaDict = collections.OrderedDict ()

ct = 0
for ep in sigma_es :
    print ('Starting simulation for: ' + str ((ct, ep)))

    epsilon_e = c_error * np.sqrt ((L * sigma_a**2) + ep**2) # conditional error threshold
    
    R2, nMax, nPaths, Vy, A, E, Y, Xref, Muts = rmf_simulation_wrapper (nsims, M, sigma_a, ep, epsilon=epsilon_e, seed=seed,
                                                                        scenarios=scenarios, weighted=weighted)

    SigmaDict[ct] = dict ()
    SigmaDict[ct]['R2']       = cp.deepcopy (R2)
    SigmaDict[ct]['nMax']     = cp.deepcopy (nMax)
    SigmaDict[ct]['nPaths']   = cp.deepcopy (nPaths)
    SigmaDict[ct]['Am'] = cp.deepcopy (Vy)
    SigmaDict[ct]['A']  = cp.deepcopy (A)
    SigmaDict[ct]['E']  = cp.deepcopy (E)
    SigmaDict[ct]['X']  = cp.deepcopy (Xref)
    SigmaDict[ct]['Y']  = cp.deepcopy (Y)
    SigmaDict[ct]['Mutations'] = cp.deepcopy (Muts)
    SigmaDict[ct]['sigmae'] = ep

    ct += 1

Starting simulation for: (0, np.float64(0.1))
Starting simulation: 0
Attempt 1 failed: Target fitness not found.
Attempt 2 failed: Target fitness not found.
Attempt 3 failed: Target fitness not found.
Attempt 4 failed: Target fitness not found.
Attempt 5 failed: Target fitness not found.
Attempt 6 failed: Target fitness not found.
Attempt 7 failed: Target fitness not found.
Attempt 8 failed: Target fitness not found.
Attempt 9 failed: Target fitness not found.
Attempt 10 failed: Target fitness not found.
Attempt 11 failed: Target fitness not found.
Attempt 12 failed: Target fitness not found.
Attempt 13 failed: Target fitness not found.
Attempt 14 failed: Target fitness not found.
Attempt 15 failed: Target fitness not found.
Attempt 16 failed: Target fitness not found.
Attempt 17 failed: Target fitness not found.
Attempt 18 failed: Target fitness not found.
Attempt 19 failed: Target fitness not found.
Attempt 20 failed: Target fitness not found.
Target fitness not found.
failure: 15
St

In [10]:
with open( os.path.join (outputdir, 'RMF_dictionary_' + str (L) + '.pkl'), 'wb') as file:
    pickle.dump(SigmaDict, file)